## TE Peak Value vs CAMELS Attributes - 52 Attributes, 4 Forcings, 671 Basins

## Load the TE shuffle results CSV and check structure

In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
import os
import pandas as pd


In [2]:
# Load the TE shuffle results
te_path = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\ALL CAMELS BASIN (FORCINGS TO STREAMFLOW)\outputs\summary_csv\TE_shuffle_all671_tau1_30_PRCPspecial_bins5.csv"

te_df = pd.read_csv(te_path, dtype={"gauge_id": str})
te_df["gauge_id"] = te_df["gauge_id"].str.zfill(8)

print("Shape:", te_df.shape)
print("\nColumns:", te_df.columns.tolist())
print("\nFirst 5 rows:")
print(te_df.head())
print("\nUnique sources:", te_df["Source"].unique())
print("Unique lags:", sorted(te_df["Lag"].unique()))
print("\nSig value counts:")
print(te_df["sig"].value_counts())

Shape: (80520, 10)

Columns: ['gauge_id', 'huc_02', 'Source', 'Target', 'Bins', 'Lag', 'TE_obs', 'thr95', 'pval', 'sig']

First 5 rows:
   gauge_id  huc_02 Source Target  Bins  Lag    TE_obs     thr95      pval  \
0  01013500       1   PRCP      Q     5    1  0.002761  0.001187  0.004975   
1  01013500       1   PRCP      Q     5    2  0.001754  0.001171  0.004975   
2  01013500       1   PRCP      Q     5    3  0.001439  0.001204  0.009950   
3  01013500       1   PRCP      Q     5    4  0.000913  0.001174  0.258706   
4  01013500       1   PRCP      Q     5    5  0.001240  0.001064  0.024876   

     sig  
0   True  
1   True  
2   True  
3  False  
4   True  

Unique sources: ['PRCP' 'SRAD' 'Tair' 'VP']
Unique lags: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20),

## Filter significant TE rows and compute peak TE and peak lag per basin per forcing

In [3]:
# keep only significant rows
te_sig = te_df[te_df["sig"] == True].copy()

print("Significant rows:", len(te_sig))

# for each basin + forcing, find the lag with the highest TE_obs
peak_idx = te_sig.groupby(["gauge_id", "Source"])["TE_obs"].idxmax()
peak_df = te_sig.loc[peak_idx, ["gauge_id", "Source", "Lag", "TE_obs"]].copy()
peak_df.columns = ["gauge_id", "Source", "peak_lag", "peak_TE"]

print("\nPeak DF shape:", peak_df.shape)
print("\nFirst 10 rows:")
print(peak_df.head(10))

# check how many basins have peak TE for each forcing
print("\nBasin count per forcing:")
print(peak_df.groupby("Source")["gauge_id"].count())

Significant rows: 34511

Peak DF shape: (2493, 4)

First 10 rows:
     gauge_id Source  peak_lag   peak_TE
0    01013500   PRCP         1  0.002761
42   01013500   SRAD        13  0.003063
73   01013500   Tair        14  0.004888
111  01013500     VP        22  0.002685
120  01022500   PRCP         1  0.016681
150  01022500   SRAD         1  0.004775
180  01022500   Tair         1  0.008300
210  01022500     VP         1  0.006192
240  01030500   PRCP         1  0.013428
274  01030500   SRAD         5  0.004072

Basin count per forcing:
Source
PRCP    670
SRAD    653
Tair    593
VP      577
Name: gauge_id, dtype: int64


## Pivot to wide format — one row per basin

In [4]:
# pivot peak_TE to wide format
peak_TE_wide = peak_df.pivot(index="gauge_id", columns="Source", values="peak_TE")
peak_TE_wide.columns = ["peak_TE_" + col for col in peak_TE_wide.columns]

# pivot peak_lag to wide format
peak_lag_wide = peak_df.pivot(index="gauge_id", columns="Source", values="peak_lag")
peak_lag_wide.columns = ["peak_lag_" + col for col in peak_lag_wide.columns]

# merge both together
peak_wide = peak_TE_wide.join(peak_lag_wide).reset_index()

print("Wide format shape:", peak_wide.shape)
print("\nColumns:", peak_wide.columns.tolist())
print("\nFirst 3 rows:")
print(peak_wide.head(3))
print("\nMissing values per column:")
print(peak_wide.isnull().sum())

Wide format shape: (671, 9)

Columns: ['gauge_id', 'peak_TE_PRCP', 'peak_TE_SRAD', 'peak_TE_Tair', 'peak_TE_VP', 'peak_lag_PRCP', 'peak_lag_SRAD', 'peak_lag_Tair', 'peak_lag_VP']

First 3 rows:
   gauge_id  peak_TE_PRCP  peak_TE_SRAD  peak_TE_Tair  peak_TE_VP  \
0  01013500      0.002761      0.003063      0.004888    0.002685   
1  01022500      0.016681      0.004775      0.008300    0.006192   
2  01030500      0.013428      0.004072      0.007460    0.005949   

   peak_lag_PRCP  peak_lag_SRAD  peak_lag_Tair  peak_lag_VP  
0            1.0           13.0           14.0         22.0  
1            1.0            1.0            1.0          1.0  
2            1.0            5.0            2.0          2.0  

Missing values per column:
gauge_id          0
peak_TE_PRCP      1
peak_TE_SRAD     18
peak_TE_Tair     78
peak_TE_VP       94
peak_lag_PRCP     1
peak_lag_SRAD    18
peak_lag_Tair    78
peak_lag_VP      94
dtype: int64


## Load CAMELS attributes and merge with peak TE table

In [5]:
# load attributes
attr_path = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\camels_attributes_combined_671basins.csv"

attr_df = pd.read_csv(attr_path, dtype={"gauge_id": str})
attr_df["gauge_id"] = attr_df["gauge_id"].str.zfill(8)

print("Attributes shape:", attr_df.shape)
print("\nFirst 3 rows:")
print(attr_df.head(3))
print("\nColumns:")
print(attr_df.columns.tolist())

Attributes shape: (671, 60)

First 3 rows:
   gauge_id  huc_02                                   gauge_name    p_mean  \
0  01013500       1             Fish River near Fort Kent, Maine  3.126679   
1  01022500       1      Narraguagus River at Cherryfield, Maine  3.608126   
2  01030500       1  Mattawamkeag River near Mattawamkeag, Maine  3.274405   

   pet_mean  p_seasonality  frac_snow   aridity  high_prec_freq  \
0  1.971555       0.187940   0.313440  0.630559           12.95   
1  2.119256      -0.114530   0.245259  0.587356           20.55   
2  2.043594       0.047358   0.277018  0.624111           17.15   

   high_prec_dur  ... area_geospa_fabric  frac_forest   lai_max  lai_diff  \
0       1.348958  ...            2303.95       0.9063  4.167304  3.340732   
1       1.205279  ...             620.38       0.9232  4.871392  3.746692   
2       1.207746  ...            3676.09       0.8782  4.685200  3.665543   

    gvf_max  gvf_diff dom_land_cover_frac     dom_land_cover  root

## Merge attributes with peak TE table and identify numeric attributes

In [6]:
# merge peak TE wide table with attributes
merged = pd.merge(peak_wide, attr_df, on="gauge_id", how="inner")
print("Merged shape:", merged.shape)

# drop columns that are not numeric or not useful for scatter plots
drop_cols = ["gauge_id", "huc_02", "gauge_name", "dom_land_cover",
             "geol_1st_class", "geol_2nd_class"]

# get numeric attribute columns only
attr_cols = [c for c in attr_df.columns if c not in drop_cols]

# check NaN count per attribute
nan_counts = merged[attr_cols].isnull().sum()
print("\nNaN count per attribute:")
print(nan_counts[nan_counts > 0])

print("\nTotal numeric attributes:", len(attr_cols))
print("\nAttribute list:")
print(attr_cols)

Merged shape: (671, 68)

NaN count per attribute:
geol_porostiy     3
q_mean            1
runoff_ratio      1
slope_fdc         1
stream_elas       1
q5                1
q95               1
high_q_freq       1
high_q_dur        1
low_q_freq        1
low_q_dur         1
zero_q_freq       1
hfd_mean          1
root_depth_50    24
root_depth_99    24
dtype: int64

Total numeric attributes: 54

Attribute list:
['p_mean', 'pet_mean', 'p_seasonality', 'frac_snow', 'aridity', 'high_prec_freq', 'high_prec_dur', 'high_prec_timing', 'low_prec_freq', 'low_prec_dur', 'low_prec_timing', 'glim_1st_class_frac', 'glim_2nd_class_frac', 'carbonate_rocks_frac', 'geol_porostiy', 'geol_permeability', 'q_mean', 'runoff_ratio', 'slope_fdc', 'baseflow_index', 'stream_elas', 'q5', 'q95', 'high_q_freq', 'high_q_dur', 'low_q_freq', 'low_q_dur', 'zero_q_freq', 'hfd_mean', 'soil_depth_pelletier', 'soil_depth_statsgo', 'soil_porosity', 'soil_conductivity', 'max_water_content', 'sand_frac', 'silt_frac', 'clay_frac',

## Finalize the attribute list — remove lat/lon

In [7]:
# remove lat and lon — they are coordinates, not hydrologic attributes
attr_cols = [c for c in attr_cols if c not in ["gauge_lat", "gauge_lon"]]

# define which attributes are categorical
cat_attrs = ["high_prec_timing", "low_prec_timing"]
season_order = ["djf", "mam", "jja", "son"]

print("Final attribute count:", len(attr_cols))
print("Categorical attributes:", cat_attrs)

Final attribute count: 52
Categorical attributes: ['high_prec_timing', 'low_prec_timing']


## Plot Peak TE vs Attributes — 52-page PDF (4 forcings per page, colored by peak lag group)

In [8]:
#  lag color groups
def get_lag_group(lag):
    if lag <= 2:
        return 0
    elif lag <= 6:
        return 1
    elif lag <= 11:
        return 2
    elif lag <= 18:
        return 3
    else:
        return 4

lag_group_labels = ["Lag 1–2", "Lag 3–6", "Lag 7–11", "Lag 12–18", "Lag 19–30"]
lag_colors = ["#e41a1c", "#ff7f00", "#4daf4a", "#377eb8", "#984ea3"]
season_order = ["djf", "mam", "jja", "son"]
forcings = ["PRCP", "SRAD", "Tair", "VP"]

out_pdf = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\pdf\PeakTE_vs_Attributes_52pages.pdf"

os.makedirs(os.path.dirname(out_pdf), exist_ok=True)

with PdfPages(out_pdf) as pdf:
    for attr in attr_cols:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"Peak TE vs  {attr}", fontsize=14, fontweight="bold")

        for ax, forcing in zip(axes, forcings):
            te_col  = f"peak_TE_{forcing}"
            lag_col = f"peak_lag_{forcing}"

            sub = merged[[attr, te_col, lag_col]].dropna()

            # CATEGORICAL (box + strip)
            if attr in cat_attrs:
                sub[attr] = sub[attr].str.lower().str.strip()
                sub = sub[sub[attr].isin(season_order)]

                box_data = [sub[sub[attr] == s][te_col].values for s in season_order]
                bp = ax.boxplot(box_data, positions=range(4), widths=0.4,
                                patch_artist=True,
                                boxprops=dict(facecolor="lightgrey", color="grey"),
                                medianprops=dict(color="black", linewidth=1.5),
                                whiskerprops=dict(color="grey"),
                                capprops=dict(color="grey"),
                                flierprops=dict(marker="", linestyle="none"))

                for si, season in enumerate(season_order):
                    s_sub = sub[sub[attr] == season]
                    y_vals = s_sub[te_col].values
                    lags   = s_sub[lag_col].values
                    jitter = np.random.uniform(-0.15, 0.15, size=len(y_vals))
                    for g in range(5):
                        mask = np.array([get_lag_group(l) == g for l in lags])
                        if mask.sum() > 0:
                            ax.scatter(si + jitter[mask], y_vals[mask],
                                       color=lag_colors[g], alpha=0.6,
                                       s=15, zorder=3)

                ax.set_xticks(range(4))
                ax.set_xticklabels(season_order, fontsize=8)
                ax.set_title(f"{forcing}", fontsize=10)
                ax.set_xlabel("Season", fontsize=8)

            # NUMERIC (scatter + regression)
            else:
                x = sub[attr].values
                y = sub[te_col].values
                lags = sub[lag_col].values

                try:
                    x = x.astype(float)
                    y = y.astype(float)
                except:
                    ax.set_title(f"{forcing}  |  skipped", fontsize=10)
                    continue

                valid = np.isfinite(x) & np.isfinite(y)
                x, y, lags = x[valid], y[valid], lags[valid]

                for g in range(5):
                    mask = np.array([get_lag_group(l) == g for l in lags])
                    if mask.sum() > 0:
                        ax.scatter(x[mask], y[mask],
                                   color=lag_colors[g], alpha=0.6,
                                   s=15, label=lag_group_labels[g])

                # fixed: added np.std(x) > 0 check and added n to title
                if len(x) > 2 and np.std(x) > 0:
                    slope, intercept, r, p, _ = stats.linregress(x, y)
                    x_line = np.linspace(x.min(), x.max(), 100)
                    y_line = slope * x_line + intercept
                    ax.plot(x_line, y_line, color="black",
                            linewidth=1.2, linestyle="--")
                    ax.set_title(f"{forcing}  |  r={r:.2f}  |  n={len(x)}", fontsize=10)

                ax.set_xlabel(attr, fontsize=8)

            ax.set_ylabel("Peak TE", fontsize=8)
            ax.tick_params(labelsize=7)

        # shared legend
        handles = [mpatches.Patch(color=lag_colors[g], label=lag_group_labels[g])
                   for g in range(5)]
        fig.legend(handles=handles, loc="lower center", ncol=5,
                   fontsize=8, title="Peak Lag Group", title_fontsize=8,
                   bbox_to_anchor=(0.5, -0.04))

        plt.tight_layout(rect=[0, 0.04, 1, 1])
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("Done! PDF saved to:")
print(out_pdf)

Done! PDF saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\pdf\PeakTE_vs_Attributes_52pages.pdf


In [9]:
print("Basin count per forcing:")
print(peak_df.groupby("Source")["gauge_id"].count())

Basin count per forcing:
Source
PRCP    670
SRAD    653
Tair    593
VP      577
Name: gauge_id, dtype: int64


## Plot Peak Lag vs Attributes — 52-page PDF (4 forcings per page)

Same layout as Step 7 but now the y-axis is peak lag instead of peak TE.
This helps us understand which basin attributes are linked to short or long lags.
For example, why does Tair peak at lag 21 in some basins?
We look for attributes like frac_snow, elev_mean, or slope_mean that may explain it.

In [10]:
def get_lag_group(lag):
    if lag <= 2:
        return 0
    elif lag <= 6:
        return 1
    elif lag <= 11:
        return 2
    elif lag <= 18:
        return 3
    else:
        return 4

lag_group_labels = ["Lag 1–2", "Lag 3–6", "Lag 7–11", "Lag 12–18", "Lag 19–30"]
lag_colors       = ["#e41a1c", "#ff7f00", "#4daf4a", "#377eb8", "#984ea3"]
forcings         = ["PRCP", "SRAD", "Tair", "VP"]

out_pdf = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\pdf\PeakLag_vs_Attributes_52pages.pdf"

os.makedirs(os.path.dirname(out_pdf), exist_ok=True)

with PdfPages(out_pdf) as pdf:
    for attr in attr_cols:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"Peak Lag vs  {attr}", fontsize=14, fontweight="bold")

        for ax, forcing in zip(axes, forcings):
            te_col  = f"peak_TE_{forcing}"
            lag_col = f"peak_lag_{forcing}"

            sub = merged[[attr, lag_col, te_col]].dropna()

            # CATEGORICAL (boxplot + colored jitter)
            if attr in cat_attrs:
                sub = sub.copy()
                sub[attr] = sub[attr].str.lower().str.strip()
                sub = sub[sub[attr].isin(season_order)]

                box_data = [sub[sub[attr] == s][lag_col].values
                            for s in season_order]
                ax.boxplot(box_data, positions=range(4), widths=0.4,
                           patch_artist=True,
                           boxprops=dict(facecolor="lightgrey", color="grey"),
                           medianprops=dict(color="black", linewidth=1.5),
                           whiskerprops=dict(color="grey"),
                           capprops=dict(color="grey"),
                           flierprops=dict(marker="", linestyle="none"))

                # colored jitter points on top
                for si, season in enumerate(season_order):
                    s_sub  = sub[sub[attr] == season]
                    y_vals = s_sub[lag_col].values
                    lags   = s_sub[lag_col].values
                    jitter = np.random.uniform(-0.15, 0.15, size=len(y_vals))
                    for g in range(5):
                        mask = np.array([get_lag_group(l) == g for l in lags])
                        if mask.sum() > 0:
                            ax.scatter(si + jitter[mask], y_vals[mask],
                                       color=lag_colors[g], alpha=0.6,
                                       s=15, zorder=3)

                ax.set_xticks(range(4))
                ax.set_xticklabels(season_order, fontsize=8)
                ax.set_title(f"{forcing}", fontsize=10)
                ax.set_xlabel("Season", fontsize=8)

            # NUMERIC (scatter colored by lag group + regression)
            else:
                x = sub[attr].values
                y = sub[lag_col].values

                try:
                    x = x.astype(float)
                    y = y.astype(float)
                except:
                    ax.set_title(f"{forcing}  |  skipped", fontsize=10)
                    continue

                valid = np.isfinite(x) & np.isfinite(y)
                x, y = x[valid], y[valid]
                lags  = y.copy()

                # color points by lag group
                for g in range(5):
                    mask = np.array([get_lag_group(l) == g for l in lags])
                    if mask.sum() > 0:
                        ax.scatter(x[mask], y[mask],
                                   color=lag_colors[g], alpha=0.6,
                                   s=15, label=lag_group_labels[g])

                if len(x) > 2 and np.std(x) > 0:
                    slope, intercept, r, p, _ = stats.linregress(x, y)
                    x_line = np.linspace(x.min(), x.max(), 100)
                    y_line = slope * x_line + intercept
                    ax.plot(x_line, y_line, color="black",
                            linewidth=1.2, linestyle="--")
                    ax.set_title(f"{forcing}  |  r={r:.2f}  |  n={len(x)}",
                                 fontsize=10)

                ax.set_xlabel(attr, fontsize=8)

            ax.set_ylabel("Peak Lag (days)", fontsize=8)
            ax.tick_params(labelsize=7)

        # shared legend
        handles = [mpatches.Patch(color=lag_colors[g], label=lag_group_labels[g])
                   for g in range(5)]
        fig.legend(handles=handles, loc="lower center", ncol=5,
                   fontsize=8, title="Peak Lag Group", title_fontsize=8,
                   bbox_to_anchor=(0.5, -0.04))

        plt.tight_layout(rect=[0, 0.04, 1, 1])
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("Done! PDF saved to:")
print(out_pdf)

Done! PDF saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\pdf\PeakLag_vs_Attributes_52pages.pdf


## Interpretation of Peak Lag vs CAMELS Attributes

### What we found

**frac_snow, elev_mean, and slope_mean — opposite effect on PRCP vs Tair/VP lag.**
Higher snow fraction and higher elevation → longer PRCP peak lag (r ~ +0.27 to +0.33).
But higher snow fraction and elevation → shorter Tair and VP peak lag (r ~ -0.30 to -0.41).
This pattern makes physical sense:
- In snow-dominated basins, rainfall does not directly produce runoff — it may be stored
  as snowpack first, so the PRCP lag is longer.
- Temperature and vapor pressure drive snowmelt more directly during the melt season,
  so Tair and VP reach their peak TE at shorter lags in those basins.
slope_mean shows the same direction but weaker for PRCP (+0.09) and moderate for Tair (-0.34).

**p_mean and runoff_ratio — wetter basins have shorter lags.**
Higher mean precipitation and higher runoff ratio → shorter peak lag for PRCP and SRAD
(r ~ -0.20 to -0.21).
Wetter basins respond faster — rainfall quickly produces runoff, so the lag is short.
Tair and VP show weaker relationship with p_mean and runoff_ratio.

**aridity — drier basins have longer PRCP lag.**
Higher aridity → longer PRCP lag (r = +0.26).
In dry basins, rainfall takes longer to produce a streamflow response because more water
is lost to evaporation or absorbed into dry soils before reaching the stream.
Tair and VP show near-zero r with aridity, meaning aridity does not explain their lag well.

**p_seasonality — stronger effect on SRAD, Tair, and VP lag.**
Higher p_seasonality (summer-dominated rainfall) → longer SRAD, Tair, and VP lag
(r ~ +0.19 to +0.33).
When precipitation is concentrated in summer (same season as peak radiation and temperature),
these energy-related forcings take longer to produce a streamflow response —
possibly because the energy and water signals overlap seasonally, creating a delayed response.
PRCP lag shows almost no relationship (r = +0.04).

**hfd_mean — splits PRCP from Tair/VP clearly.**
Later half-flow date (more summer-dominated flow) → longer PRCP lag (r = +0.22)
but shorter Tair and VP lag (r = -0.28 to -0.31).
Basins where streamflow peaks in summer tend to be snowmelt-driven —
temperature and vapor pressure govern that seasonal melt pattern at relatively short lags,
while direct rainfall is a less efficient driver of summer streamflow.

**high_prec_freq — strongest effect on Tair lag.**
More frequent heavy rainfall events → longer Tair lag (r = +0.30).
When heavy rainfall is frequent, the temperature signal takes longer to emerge above the
rainfall-driven variability in streamflow, pushing the peak Tair TE to longer lags.

**baseflow_index — modest negative effect on Tair lag only.**
Higher baseflow index → slightly shorter Tair lag (r = -0.18).
Basins with more groundwater contribution may integrate temperature signals more slowly,
but the relationship is weak.

### Summary of strongest lag signals

| Attribute      | Forcing with strongest r | r value | Direction |
|---------------|--------------------------|---------|-----------|
| frac_snow     | Tair                     | -0.41   | negative  |
| elev_mean     | PRCP                     | +0.33   | positive  |
| p_seasonality | SRAD                     | +0.33   | positive  |
| slope_mean    | Tair                     | -0.34   | negative  |
| elev_mean     | Tair                     | -0.33   | negative  |
| high_prec_freq| Tair                     | +0.30   | positive  |
| hfd_mean      | VP                       | -0.31   | negative  |
| frac_snow     | VP                       | -0.32   | negative  |


# Correlation Summary CSV — r values for Peak TE and Peak Lag vs all 52 attributes

We collect all r values from the scatter plots into one CSV table.
This makes it easy to rank which attributes matter most for each forcing,
and to compare peak TE vs peak lag patterns side by side.

In [11]:
forcings = ["PRCP", "SRAD", "Tair", "VP"]

results = []

for attr in attr_cols:
    if attr in cat_attrs:
        continue  # skip categorical attributes

    for forcing in forcings:
        te_col  = f"peak_TE_{forcing}"
        lag_col = f"peak_lag_{forcing}"

        #  peak TE vs attribute
        sub_te = merged[[attr, te_col]].dropna()
        if len(sub_te) > 2:
            try:
                x = sub_te[attr].astype(float).values
                y = sub_te[te_col].astype(float).values
                valid = np.isfinite(x) & np.isfinite(y)
                x, y = x[valid], y[valid]
                _, _, r_te, _, _ = stats.linregress(x, y)
                n_te = len(x)
            except:
                r_te, n_te = np.nan, np.nan
        else:
            r_te, n_te = np.nan, np.nan

        #  peak lag vs attribute
        sub_lag = merged[[attr, lag_col]].dropna()
        if len(sub_lag) > 2:
            try:
                x = sub_lag[attr].astype(float).values
                y = sub_lag[lag_col].astype(float).values
                valid = np.isfinite(x) & np.isfinite(y)
                x, y = x[valid], y[valid]
                _, _, r_lag, _, _ = stats.linregress(x, y)
                n_lag = len(x)
            except:
                r_lag, n_lag = np.nan, np.nan
        else:
            r_lag, n_lag = np.nan, np.nan

        results.append({
            "attribute"  : attr,
            "forcing"    : forcing,
            "r_peak_TE"  : round(r_te,  3),
            "n_peak_TE"  : n_te,
            "r_peak_lag" : round(r_lag, 3),
            "n_peak_lag" : n_lag
        })

corr_df = pd.DataFrame(results)

# save
out_csv = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\correlation_summary_TE_lag_vs_attributes.csv"

os.makedirs(os.path.dirname(out_csv), exist_ok=True)
corr_df.to_csv(out_csv, index=False)

print("Shape:", corr_df.shape)
print("\nFirst 10 rows:")
print(corr_df.head(10))
print("\nSaved to:", out_csv)

Shape: (200, 6)

First 10 rows:
       attribute forcing  r_peak_TE  n_peak_TE  r_peak_lag  n_peak_lag
0         p_mean    PRCP      0.588        670      -0.212         670
1         p_mean    SRAD      0.361        653      -0.211         653
2         p_mean    Tair      0.217        593      -0.064         593
3         p_mean      VP      0.151        577       0.080         577
4       pet_mean    PRCP     -0.191        670       0.108         670
5       pet_mean    SRAD     -0.139        653      -0.186         653
6       pet_mean    Tair     -0.216        593       0.123         593
7       pet_mean      VP     -0.212        577       0.027         577
8  p_seasonality    PRCP     -0.366        670       0.039         670
9  p_seasonality    SRAD     -0.480        653       0.327         653

Saved to: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\

## Rank attributes by correlation strength — top 10 for each forcing

We sort the correlation table by absolute r value to find which attributes
are most strongly linked to peak TE and peak lag for each forcing.
This makes it easy to summarize the key findings.

In [12]:
# add absolute r columns
corr_df["abs_r_peak_TE"]  = corr_df["r_peak_TE"].abs()
corr_df["abs_r_peak_lag"] = corr_df["r_peak_lag"].abs()

forcings = ["PRCP", "SRAD", "Tair", "VP"]

print("=" * 60)
print("TOP 10 ATTRIBUTES BY |r| FOR PEAK TE")
print("=" * 60)
for f in forcings:
    sub = corr_df[corr_df["forcing"] == f].sort_values("abs_r_peak_TE", ascending=False).head(10)
    print(f"\n--- {f} ---")
    print(sub[["attribute", "r_peak_TE"]].to_string(index=False))

print("\n")
print("=" * 60)
print("TOP 10 ATTRIBUTES BY |r| FOR PEAK LAG")
print("=" * 60)
for f in forcings:
    sub = corr_df[corr_df["forcing"] == f].sort_values("abs_r_peak_lag", ascending=False).head(10)
    print(f"\n--- {f} ---")
    print(sub[["attribute", "r_peak_lag"]].to_string(index=False))

TOP 10 ATTRIBUTES BY |r| FOR PEAK TE

--- PRCP ---
     attribute  r_peak_TE
           q95      0.624
        q_mean      0.612
        p_mean      0.588
  runoff_ratio      0.441
 low_prec_freq     -0.405
            q5      0.399
high_prec_freq     -0.368
 p_seasonality     -0.366
      hfd_mean     -0.360
       aridity     -0.337

--- SRAD ---
     attribute  r_peak_TE
           q95      0.603
  runoff_ratio      0.574
        q_mean      0.517
    slope_mean      0.505
 p_seasonality     -0.480
high_prec_freq     -0.480
 low_prec_freq     -0.404
     frac_snow      0.382
      gvf_diff     -0.372
        p_mean      0.361

--- Tair ---
     attribute  r_peak_TE
  runoff_ratio      0.536
high_prec_freq     -0.512
     frac_snow      0.480
           q95      0.455
 low_prec_freq     -0.448
    slope_mean      0.436
        q_mean      0.382
   stream_elas     -0.360
     elev_mean      0.342
 p_seasonality     -0.338

--- VP ---
     attribute  r_peak_TE
     frac_snow      0.480

## Interpretation of Ranked Correlations — Peak TE and Peak Lag

### Peak TE — what makes TE stronger or weaker?

**PRCP:** Top predictors are q95 (+0.62), q_mean (+0.61), p_mean (+0.59),
runoff_ratio (+0.44), low_prec_freq (-0.41), q5 (+0.40), high_prec_freq (-0.37),
p_seasonality (-0.37), hfd_mean (-0.36), aridity (-0.34).
Basins with more water (higher streamflow and rainfall) show stronger PRCP TE.
Both high_prec_freq and low_prec_freq are negative — meaning basins with
more frequent dry or wet events tend to have weaker PRCP TE overall.
Drier, more seasonal basins (high aridity, high p_seasonality) also show weaker PRCP TE.

**SRAD:** Top predictors are q95 (+0.60), runoff_ratio (+0.57), q_mean (+0.52),
slope_mean (+0.51), p_seasonality (-0.48), high_prec_freq (-0.48),
low_prec_freq (-0.40), frac_snow (+0.38), gvf_diff (-0.37), p_mean (+0.36).
Water availability still matters but slope_mean and frac_snow also appear strongly.
Steeper and snowier basins show stronger SRAD TE.
Highly seasonal and frequently wet or dry basins show weaker SRAD TE.

**Tair:** Top predictors are runoff_ratio (+0.54), high_prec_freq (-0.51),
frac_snow (+0.48), q95 (+0.46), low_prec_freq (-0.45), slope_mean (+0.44),
q_mean (+0.38), stream_elas (-0.36), elev_mean (+0.34), p_seasonality (-0.34).
Snow fraction, slope, and elevation become more prominent here alongside water availability.
High precipitation frequency is the strongest negative predictor for Tair TE —
when rainfall is very frequent, the temperature signal in streamflow is harder to detect.

**VP:** Top predictors are frac_snow (+0.48), runoff_ratio (+0.48),
high_prec_freq (-0.45), low_prec_freq (-0.40), slope_mean (+0.38), q95 (+0.36),
elev_mean (+0.34), stream_elas (-0.31), q_mean (+0.30), p_seasonality (-0.25).
VP is most similar to Tair — frac_snow is the top predictor,
and high/low precipitation frequency both reduce VP TE.

**Key pattern across all forcings:**
Water availability (q95, q_mean, runoff_ratio) is the most consistent positive predictor
for all four forcings. For energy forcings (SRAD, Tair, VP), snow fraction and slope
additionally emerge as strong positive predictors — pointing to snowmelt as the
key physical mechanism. Both high_prec_freq and low_prec_freq are consistently
negative across all forcings, meaning basins with more frequent rainfall events
(wet or dry) tend to show weaker TE signals overall.

---

### Peak Lag — what makes lags shorter or longer?

**PRCP:** Top predictors are elev_mean (+0.33), frac_snow (+0.27), aridity (+0.26),
hfd_mean (+0.22), p_mean (-0.21), root_depth_50 (-0.21), gvf_max (-0.20),
lai_max (-0.18), high_q_dur (+0.17), slope_fdc (-0.15).
Higher elevation, more snow, and higher aridity all increase PRCP lag.
In dry and snowy basins, rainfall takes longer to reach the stream —
either lost to dry soils or stored temporarily as snow.
Wetter basins (higher p_mean) and basins with more vegetation (gvf_max, lai_max)
have shorter PRCP lags.

**SRAD:** Top predictors are p_seasonality (+0.33), gvf_diff (+0.31),
silt_frac (+0.23), q95 (-0.22), slope_mean (-0.21), p_mean (-0.21),
q_mean (-0.20), runoff_ratio (-0.20), q5 (-0.19), pet_mean (-0.19).
Longer SRAD lag is linked to stronger precipitation seasonality and
higher vegetation seasonality (gvf_diff).
When both rainfall and vegetation peak strongly in one season,
the radiation signal takes longer to emerge in streamflow.
Wetter basins (q95, q_mean, runoff_ratio, p_mean) all show shorter SRAD lags.

**Tair:** Top predictors are frac_snow (-0.41), slope_mean (-0.34),
elev_mean (-0.33), high_prec_freq (+0.30), runoff_ratio (-0.29),
hfd_mean (-0.28), glim_1st_class_frac (+0.27), q95 (-0.25),
gvf_diff (+0.24), q_mean (-0.22).
This is the clearest pattern of all four forcings.
Higher snow fraction, steeper slope, and higher elevation all shorten Tair lag.
In snow-dominated mountain basins, streamflow responds to temperature quickly
during the melt season — so the peak Tair TE appears at a short lag.
More frequent heavy rainfall (high_prec_freq +0.30) increases Tair lag —
in rainfall-dominated basins, the temperature signal takes longer to emerge.

**VP:** Top predictors are frac_snow (-0.32), elev_mean (-0.32),
lai_diff (+0.32), high_prec_dur (-0.32), hfd_mean (-0.31), slope_mean (-0.30),
root_depth_50 (+0.29), lai_max (+0.28), high_q_dur (-0.26), gvf_max (+0.26).
Similar to Tair — snow and elevation shorten VP lag.
Higher vegetation seasonality (lai_diff, gvf_max) and deeper roots (root_depth_50)
increase VP lag, possibly because vegetation mediates the vapor pressure
signal to streamflow through transpiration, introducing additional delay.

---

### Overall summary

| Forcing | Strongest Peak TE predictor | Strongest Peak Lag predictor |
|---------|----------------------------|------------------------------|
| PRCP    | q95 (+0.62)                | elev_mean (+0.33)            |
| SRAD    | q95 (+0.60)                | p_seasonality (+0.33)        |
| Tair    | runoff_ratio (+0.54)       | frac_snow (-0.41)            |
| VP      | frac_snow (+0.48)          | frac_snow (-0.32)            |

**frac_snow** is a key attribute across energy forcings.
It increases peak TE strength for SRAD, Tair, and VP (more snowmelt signal),
and at the same time shortens their peak lag (snowmelt responds quickly to temperature).
This is a consistent snowmelt signature in the TE results.


## Find dominant forcing per basin — one point per basin

For each basin, we look at the peak TE values across all forcings
that passed the shuffle significance test.
The forcing with the highest peak TE is the dominant forcing for that basin.
Each basin gets exactly one row — its dominant forcing, peak TE, and peak lag.
Basins with no significant TE for any forcing are dropped.

In [13]:
import pandas as pd
import numpy as np

# peak_df already exists from second Step 
# it has columns: gauge_id, Source, peak_lag, peak_TE
# it contains only significant TE rows, one row per basin per forcing

# for each basin, find the forcing with the highest peak TE
dominant_idx = peak_df.groupby("gauge_id")["peak_TE"].idxmax()
dominant_df  = peak_df.loc[dominant_idx].copy()
dominant_df.columns = ["gauge_id", "dom_forcing", "dom_peak_lag", "dom_peak_TE"]
dominant_df = dominant_df.reset_index(drop=True)

print("Total basins with at least one significant TE:", len(dominant_df))
print("\nDominant forcing counts:")
print(dominant_df["dom_forcing"].value_counts())
print("\nFirst 10 rows:")
print(dominant_df.head(10))

Total basins with at least one significant TE: 671

Dominant forcing counts:
dom_forcing
PRCP    389
SRAD    156
Tair     95
VP       31
Name: count, dtype: int64

First 10 rows:
   gauge_id dom_forcing  dom_peak_lag  dom_peak_TE
0  01013500        Tair            14     0.004888
1  01022500        PRCP             1     0.016681
2  01030500        PRCP             1     0.013428
3  01031500        PRCP             1     0.003859
4  01047000        PRCP             1     0.002277
5  01052500        Tair             1     0.006612
6  01054200        Tair            12     0.003738
7  01055000        SRAD             2     0.003516
8  01057000        PRCP             1     0.002255
9  01073000        PRCP             1     0.005689


## Merge dominant forcing table with CAMELS attributes

In [14]:
# merge dominant_df with attributes
dominant_merged = pd.merge(dominant_df, attr_df, on="gauge_id", how="inner")

print("Merged shape:", dominant_merged.shape)
print("\nMissing values in key columns:")
print(dominant_merged[["dom_peak_TE", "dom_peak_lag", "dom_forcing"]].isnull().sum())

Merged shape: (671, 63)

Missing values in key columns:
dom_peak_TE     0
dom_peak_lag    0
dom_forcing     0
dtype: int64


## Plot Peak TE vs Attributes — dominant forcing only, 4 subplots per page, colored by peak lag group

Each subplot shows only basins where that forcing is dominant.
PRCP subplot: 389 basins, SRAD: 156, Tair: 95, VP: 31.
Points are colored by peak lag group same as before.

In [15]:
def get_lag_group(lag):
    if lag <= 2:
        return 0
    elif lag <= 6:
        return 1
    elif lag <= 11:
        return 2
    elif lag <= 18:
        return 3
    else:
        return 4

lag_group_labels = ["Lag 1–2", "Lag 3–6", "Lag 7–11", "Lag 12–18", "Lag 19–30"]
lag_colors       = ["#e41a1c", "#ff7f00", "#4daf4a", "#377eb8", "#984ea3"]
forcings         = ["PRCP", "SRAD", "Tair", "VP"]

out_pdf = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\pdf\DominantForcing_PeakTE_vs_Attributes_52pages.pdf"

os.makedirs(os.path.dirname(out_pdf), exist_ok=True)

with PdfPages(out_pdf) as pdf:
    for attr in attr_cols:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"Peak TE vs {attr} — dominant forcing only", fontsize=14, fontweight="bold")

        for ax, forcing in zip(axes, forcings):

            # filter to only basins where this forcing is dominant
            sub = dominant_merged[dominant_merged["dom_forcing"] == forcing][[
                attr, "dom_peak_TE", "dom_peak_lag"]].dropna()

            n = len(sub)

            # CATEGORICAL
            if attr in cat_attrs:
                sub = sub.copy()
                sub[attr] = sub[attr].str.lower().str.strip()
                sub = sub[sub[attr].isin(season_order)]

                box_data = [sub[sub[attr] == s]["dom_peak_TE"].values
                            for s in season_order]
                ax.boxplot(box_data, positions=range(4), widths=0.4,
                           patch_artist=True,
                           boxprops=dict(facecolor="lightgrey", color="grey"),
                           medianprops=dict(color="black", linewidth=1.5),
                           whiskerprops=dict(color="grey"),
                           capprops=dict(color="grey"),
                           flierprops=dict(marker="", linestyle="none"))

                for si, season in enumerate(season_order):
                    s_sub  = sub[sub[attr] == season]
                    y_vals = s_sub["dom_peak_TE"].values
                    lags   = s_sub["dom_peak_lag"].values
                    jitter = np.random.uniform(-0.15, 0.15, size=len(y_vals))
                    for g in range(5):
                        mask = np.array([get_lag_group(l) == g for l in lags])
                        if mask.sum() > 0:
                            ax.scatter(si + jitter[mask], y_vals[mask],
                                       color=lag_colors[g], alpha=0.6,
                                       s=15, zorder=3)

                ax.set_xticks(range(4))
                ax.set_xticklabels(season_order, fontsize=8)
                ax.set_title(f"{forcing}  |  n={n}", fontsize=10)
                ax.set_xlabel("Season", fontsize=8)

            # NUMERIC
            else:
                try:
                    x = sub[attr].astype(float).values
                except:
                    ax.set_title(f"{forcing}  |  skipped", fontsize=10)
                    continue

                y    = sub["dom_peak_TE"].astype(float).values
                lags = sub["dom_peak_lag"].values

                valid = np.isfinite(x) & np.isfinite(y)
                x, y, lags = x[valid], y[valid], lags[valid]

                for g in range(5):
                    mask = np.array([get_lag_group(l) == g for l in lags])
                    if mask.sum() > 0:
                        ax.scatter(x[mask], y[mask],
                                   color=lag_colors[g], alpha=0.6,
                                   s=15, label=lag_group_labels[g])

                if len(x) > 2 and np.std(x) > 0:
                    slope, intercept, r, p, _ = stats.linregress(x, y)
                    x_line = np.linspace(x.min(), x.max(), 100)
                    y_line = slope * x_line + intercept
                    ax.plot(x_line, y_line, color="black",
                            linewidth=1.2, linestyle="--")
                    ax.set_title(f"{forcing}  |  r={r:.2f}  |  n={n}", fontsize=10)
                else:
                    ax.set_title(f"{forcing}  |  n={n}", fontsize=10)

                ax.set_xlabel(attr, fontsize=8)

            ax.set_ylabel("Peak TE", fontsize=8)
            ax.tick_params(labelsize=7)

        # shared legend
        handles = [mpatches.Patch(color=lag_colors[g], label=lag_group_labels[g])
                   for g in range(5)]
        fig.legend(handles=handles, loc="lower center", ncol=5,
                   fontsize=8, title="Peak Lag Group", title_fontsize=8,
                   bbox_to_anchor=(0.5, -0.04))

        plt.tight_layout(rect=[0, 0.04, 1, 1])
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("Done! PDF saved to:")
print(out_pdf)

Done! PDF saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\pdf\DominantForcing_PeakTE_vs_Attributes_52pages.pdf


## Plot Peak Lag vs Attributes — dominant forcing only, 4 subplots per page, colored by peak lag group

Same layout as Step 14 but now y-axis is peak lag instead of peak TE.
Each subplot shows only basins where that forcing is dominant.
PRCP: 389 basins, SRAD: 156, Tair: 95, VP: 31.

In [16]:
def get_lag_group(lag):
    if lag <= 2:
        return 0
    elif lag <= 6:
        return 1
    elif lag <= 11:
        return 2
    elif lag <= 18:
        return 3
    else:
        return 4

lag_group_labels = ["Lag 1–2", "Lag 3–6", "Lag 7–11", "Lag 12–18", "Lag 19–30"]
lag_colors       = ["#e41a1c", "#ff7f00", "#4daf4a", "#377eb8", "#984ea3"]
forcings         = ["PRCP", "SRAD", "Tair", "VP"]

out_pdf = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\pdf\DominantForcing_PeakLag_vs_Attributes_52pages.pdf"

os.makedirs(os.path.dirname(out_pdf), exist_ok=True)

with PdfPages(out_pdf) as pdf:
    for attr in attr_cols:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"Peak Lag vs {attr} — dominant forcing only",
                     fontsize=14, fontweight="bold")

        for ax, forcing in zip(axes, forcings):

            # filter to only dominant basins for this forcing
            sub = dominant_merged[dominant_merged["dom_forcing"] == forcing][
                [attr, "dom_peak_lag", "dom_peak_TE"]].dropna()

            n = len(sub)

            # CATEGORICAL
            if attr in cat_attrs:
                sub = sub.copy()
                sub[attr] = sub[attr].str.lower().str.strip()
                sub = sub[sub[attr].isin(season_order)]

                box_data = [sub[sub[attr] == s]["dom_peak_lag"].values
                            for s in season_order]
                ax.boxplot(box_data, positions=range(4), widths=0.4,
                           patch_artist=True,
                           boxprops=dict(facecolor="lightgrey", color="grey"),
                           medianprops=dict(color="black", linewidth=1.5),
                           whiskerprops=dict(color="grey"),
                           capprops=dict(color="grey"),
                           flierprops=dict(marker="", linestyle="none"))

                # add colored jitter points on top of boxplot
                for si, season in enumerate(season_order):
                    s_sub  = sub[sub[attr] == season]
                    y_vals = s_sub["dom_peak_lag"].values
                    lags   = s_sub["dom_peak_lag"].values
                    jitter = np.random.uniform(-0.15, 0.15, size=len(y_vals))
                    for g in range(5):
                        mask = np.array([get_lag_group(l) == g for l in lags])
                        if mask.sum() > 0:
                            ax.scatter(si + jitter[mask], y_vals[mask],
                                       color=lag_colors[g], alpha=0.6,
                                       s=15, zorder=3)

                ax.set_xticks(range(4))
                ax.set_xticklabels(season_order, fontsize=8)
                ax.set_title(f"{forcing}  |  n={n}", fontsize=10)
                ax.set_xlabel("Season", fontsize=8)

            # NUMERIC
            else:
                try:
                    x = sub[attr].astype(float).values
                except:
                    ax.set_title(f"{forcing}  |  skipped", fontsize=10)
                    continue

                y    = sub["dom_peak_lag"].astype(float).values
                lags = sub["dom_peak_lag"].values

                valid = np.isfinite(x) & np.isfinite(y)
                x, y, lags = x[valid], y[valid], lags[valid]

                # scatter colored by lag group
                for g in range(5):
                    mask = np.array([get_lag_group(l) == g for l in lags])
                    if mask.sum() > 0:
                        ax.scatter(x[mask], y[mask],
                                   color=lag_colors[g], alpha=0.6,
                                   s=15, label=lag_group_labels[g])

                if len(x) > 2 and np.std(x) > 0:
                    slope, intercept, r, p, _ = stats.linregress(x, y)
                    x_line = np.linspace(x.min(), x.max(), 100)
                    y_line = slope * x_line + intercept
                    ax.plot(x_line, y_line, color="black",
                            linewidth=1.2, linestyle="--")
                    ax.set_title(f"{forcing}  |  r={r:.2f}  |  n={len(x)}",
                                 fontsize=10)
                else:
                    ax.set_title(f"{forcing}  |  n={n}", fontsize=10)

                ax.set_xlabel(attr, fontsize=8)

            ax.set_ylabel("Peak Lag (days)", fontsize=8)
            ax.tick_params(labelsize=7)

        # shared legend
        handles = [mpatches.Patch(color=lag_colors[g], label=lag_group_labels[g])
                   for g in range(5)]
        fig.legend(handles=handles, loc="lower center", ncol=5,
                   fontsize=8, title="Peak Lag Group", title_fontsize=8,
                   bbox_to_anchor=(0.5, -0.04))

        plt.tight_layout(rect=[0, 0.04, 1, 1])
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("Done! PDF saved to:")
print(out_pdf)

Done! PDF saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\pdf\DominantForcing_PeakLag_vs_Attributes_52pages.pdf


## Correlation summary CSV — dominant forcing only

Same as Step 10 but now using only dominant forcing basins.
Each forcing group has fewer basins:
PRCP=389, SRAD=156, Tair=95, VP=31.
We compute r between each attribute and dom_peak_TE and dom_peak_lag.

In [17]:
forcings = ["PRCP", "SRAD", "Tair", "VP"]

results_dom = []

for attr in attr_cols:
    if attr in cat_attrs:
        continue

    for forcing in forcings:

        # filter to only dominant basins for this forcing
        sub = dominant_merged[dominant_merged["dom_forcing"] == forcing][
            [attr, "dom_peak_TE", "dom_peak_lag"]].dropna()

        n = len(sub)

        # --- dom_peak_TE vs attribute ---
        if n > 2:
            try:
                x = sub[attr].astype(float).values
                y = sub["dom_peak_TE"].astype(float).values
                valid = np.isfinite(x) & np.isfinite(y)
                x, y = x[valid], y[valid]
                if len(x) > 2 and np.std(x) > 0:
                    _, _, r_te, _, _ = stats.linregress(x, y)
                    n_te = len(x)
                else:
                    r_te, n_te = np.nan, len(x)
            except:
                r_te, n_te = np.nan, np.nan
        else:
            r_te, n_te = np.nan, n

        # dom_peak_lag vs attribute
        if n > 2:
            try:
                x = sub[attr].astype(float).values
                y = sub["dom_peak_lag"].astype(float).values
                valid = np.isfinite(x) & np.isfinite(y)
                x, y = x[valid], y[valid]
                if len(x) > 2 and np.std(x) > 0:
                    _, _, r_lag, _, _ = stats.linregress(x, y)
                    n_lag = len(x)
                else:
                    r_lag, n_lag = np.nan, len(x)
            except:
                r_lag, n_lag = np.nan, np.nan
        else:
            r_lag, n_lag = np.nan, n

        results_dom.append({
            "attribute"  : attr,
            "forcing"    : forcing,
            "n_basins"   : n,
            "r_peak_TE"  : round(r_te,  3) if not np.isnan(r_te)  else np.nan,
            "n_peak_TE"  : n_te,
            "r_peak_lag" : round(r_lag, 3) if not np.isnan(r_lag) else np.nan,
            "n_peak_lag" : n_lag
        })

corr_dom_df = pd.DataFrame(results_dom)

# save
out_csv = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\correlation_summary_dominant_forcing.csv"

os.makedirs(os.path.dirname(out_csv), exist_ok=True)
corr_dom_df.to_csv(out_csv, index=False)

print("Shape:", corr_dom_df.shape)
print("\nSample — top 10 rows:")
print(corr_dom_df.head(10))
print("\nSaved to:", out_csv)

Shape: (200, 7)

Sample — top 10 rows:
       attribute forcing  n_basins  r_peak_TE  n_peak_TE  r_peak_lag  \
0         p_mean    PRCP       389      0.627        389      -0.094   
1         p_mean    SRAD       156      0.186        156      -0.225   
2         p_mean    Tair        95     -0.140         95       0.235   
3         p_mean      VP        31      0.029         31       0.542   
4       pet_mean    PRCP       389     -0.331        389       0.013   
5       pet_mean    SRAD       156     -0.082        156      -0.166   
6       pet_mean    Tair        95      0.173         95      -0.076   
7       pet_mean      VP        31     -0.143         31       0.057   
8  p_seasonality    PRCP       389     -0.370        389       0.009   
9  p_seasonality    SRAD       156     -0.545        156       0.475   

   n_peak_lag  
0         389  
1         156  
2          95  
3          31  
4         389  
5         156  
6          95  
7          31  
8         389  
9       

## Interpretation — Dominant Forcing Correlation Summary

### Important note on sample sizes
PRCP: 389 basins, SRAD: 156 basins, Tair: 95 basins, VP: 31 basins.
VP results should be interpreted with caution — 31 basins is a small sample
and r values can be unstable.

---

### Peak TE — what makes dominant TE stronger?

**PRCP (n=389):**
Top positive predictors: q95 (+0.69), q_mean (+0.66), p_mean (+0.63), runoff_ratio (+0.60).
Top negative predictors: low_prec_freq (-0.60), high_prec_freq (-0.57), aridity (-0.43),
hfd_mean (-0.39), p_seasonality (-0.37).
Same story as before — wetter basins with more runoff show stronger PRCP TE.
Both high and low precipitation frequency reduce PRCP TE in dominant basins.

**SRAD (n=156):**
Top positive predictors: slope_mean (+0.68), runoff_ratio (+0.63), q95 (+0.62),
frac_snow (+0.55), q_mean (+0.51).
Top negative predictors: high_prec_freq (-0.58), p_seasonality (-0.55),
low_prec_freq (-0.51), stream_elas (-0.43).
Slope is the strongest predictor here — steeper basins show stronger SRAD TE.
Snow and wetness also matter. Seasonal and frequently wet/dry basins show weaker SRAD TE.

**Tair (n=95):**
Top positive predictors: frac_snow (+0.71), elev_mean (+0.66), other_frac (+0.54),
runoff_ratio (+0.52), slope_mean (+0.50).
Top negative predictors: gvf_diff (-0.51), p_seasonality (-0.45), gvf_max (-0.47),
lai_diff (-0.40).
frac_snow is the dominant predictor for Tair TE — snow-dominated basins show
much stronger temperature-to-streamflow information transfer.
High vegetation seasonality (gvf_diff, lai_diff) reduces Tair TE.

**VP (n=31):**
frac_snow (+0.76) is the strongest predictor — consistent with Tair pattern.
elev_mean (+0.44) and slope_mean (+0.38) also positive.
p_seasonality (-0.50) is the strongest negative predictor.

**Key change from previous analysis (all forcings):**
When we focus only on dominant basins, the r values generally get stronger —
especially for Tair and SRAD. frac_snow for Tair goes from +0.48 to +0.71,
and slope_mean for SRAD goes from +0.51 to +0.68.
This makes sense — dominant basins are the ones where the physical mechanism
is clearest, so the attribute signal is stronger.

---

### Peak Lag — what controls lag in dominant basins?

**PRCP (n=389):**
All r values are weak (below 0.20). No single attribute strongly explains
PRCP lag in PRCP-dominant basins. This suggests PRCP lag is driven by
local factors that are not well captured by basin-average attributes.

**SRAD (n=156):**
Longer SRAD lag linked to p_seasonality (+0.48), soil_porosity (+0.36),
gvf_diff (+0.33), soil_depth_pelletier (+0.33).
Shorter SRAD lag linked to sand_frac (-0.36), runoff_ratio (-0.35),
q95 (-0.34), q_mean (-0.31).
Seasonal basins with deeper soils show longer SRAD lag.
Wetter basins with sandier soils respond faster.

**Tair (n=95):**
Strongest lag predictors of all four forcings.
frac_snow (-0.58) and hfd_mean (-0.57) — snow-dominated basins where
flow peaks early have shorter Tair lag.
elev_mean (-0.45) and slope_mean (-0.33) also shorten Tair lag.
high_prec_dur (-0.42) shortens lag too.
gvf_diff (+0.39) and glim_1st_class_frac (+0.34) increase Tair lag.
This is the clearest physical story: in cold, snowy, steep mountain basins
temperature drives streamflow quickly during melt season.

**VP (n=31):**
p_mean (+0.54) and hfd_mean (-0.46) are strongest.
clay_frac (-0.46) and geol_porostiy (-0.42) also appear.
Results are noisy given only 31 basins.

---

### Comparison: dominant forcing vs all forcings

| Forcing | frac_snow r_TE (all) | frac_snow r_TE (dominant) | Change |
|---------|----------------------|---------------------------|--------|
| PRCP    | -0.11                | +0.14                     | sign flips |
| SRAD    | +0.38                | +0.55                     | stronger |
| Tair    | +0.48                | +0.71                     | much stronger |
| VP      | +0.48                | +0.76                     | much stronger |

| Forcing | slope_mean r_TE (all) | slope_mean r_TE (dominant) | Change |
|---------|-----------------------|----------------------------|--------|
| PRCP    | +0.18                 | +0.31                      | stronger |
| SRAD    | +0.51                 | +0.68                      | stronger |
| Tair    | +0.44                 | +0.50                      | stronger |
| VP      | +0.38                 | +0.38                      | same |

Focusing on dominant basins generally strengthens the attribute-TE relationships,
especially for energy forcings (SRAD, Tair, VP).
This confirms that the physical mechanisms are cleaner and more detectable
when we look at basins where that forcing truly governs streamflow.


## Comparison table — all forcings vs dominant forcing
## r values and n for Peak TE and Peak Lag side by side, one page per forcing

In [18]:
# Step 1: build the wide comparison table

all_csv = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\correlation_summary_TE_lag_vs_attributes.csv"

dom_csv = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\correlation_summary_dominant_forcing.csv"

all_df = pd.read_csv(all_csv)
dom_df = pd.read_csv(dom_csv)

forcings   = ["PRCP", "SRAD", "Tair", "VP"]
attributes = [a for a in attr_cols if a not in cat_attrs]

rows = []
for attr in attributes:
    row = {"attribute": attr}
    for f in forcings:
        a = all_df[(all_df["attribute"] == attr) & (all_df["forcing"] == f)]
        if len(a) > 0:
            row[f"{f}_r_TE_all"]  = a["r_peak_TE"].values[0]
            row[f"{f}_n_TE_all"]  = int(a["n_peak_TE"].values[0]) if not pd.isna(a["n_peak_TE"].values[0]) else ""
            row[f"{f}_r_lag_all"] = a["r_peak_lag"].values[0]
            row[f"{f}_n_lag_all"] = int(a["n_peak_lag"].values[0]) if not pd.isna(a["n_peak_lag"].values[0]) else ""
        else:
            row[f"{f}_r_TE_all"]  = ""
            row[f"{f}_n_TE_all"]  = ""
            row[f"{f}_r_lag_all"] = ""
            row[f"{f}_n_lag_all"] = ""

        d = dom_df[(dom_df["attribute"] == attr) & (dom_df["forcing"] == f)]
        if len(d) > 0:
            row[f"{f}_r_TE_dom"]  = d["r_peak_TE"].values[0]
            row[f"{f}_n_TE_dom"]  = int(d["n_peak_TE"].values[0]) if not pd.isna(d["n_peak_TE"].values[0]) else ""
            row[f"{f}_r_lag_dom"] = d["r_peak_lag"].values[0]
            row[f"{f}_n_lag_dom"] = int(d["n_peak_lag"].values[0]) if not pd.isna(d["n_peak_lag"].values[0]) else ""
        else:
            row[f"{f}_r_TE_dom"]  = ""
            row[f"{f}_n_TE_dom"]  = ""
            row[f"{f}_r_lag_dom"] = ""
            row[f"{f}_n_lag_dom"] = ""

    rows.append(row)

wide_df = pd.DataFrame(rows)

# save CSV
out_csv = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\comparison_table_all_vs_dominant.csv"
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
wide_df.to_csv(out_csv, index=False)
print("CSV saved:", out_csv)
print("Shape:", wide_df.shape)

#  Step 2: PDF table 

out_pdf = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\pdf\comparison_table_all_vs_dominant.pdf"

os.makedirs(os.path.dirname(out_pdf), exist_ok=True)

def cell_color(val):
    try:
        v = float(val)
        if abs(v) >= 0.5:
            return "#ff9999" if v < 0 else "#66cc66"
        elif abs(v) >= 0.3:
            return "#ffccaa" if v < 0 else "#aaddaa"
        elif abs(v) >= 0.1:
            return "#fff0cc" if v < 0 else "#ddf0dd"
        else:
            return "white"
    except:
        return "white"

legend_items = [
    ("#66cc66", "|r|≥0.5\npositive"),
    ("#aaddaa", "|r|≥0.3\npositive"),
    ("#ddf0dd", "|r|≥0.1\npositive"),
    ("white",   "|r|<0.1"),
    ("#fff0cc", "|r|≥0.1\nnegative"),
    ("#ffccaa", "|r|≥0.3\nnegative"),
    ("#ff9999", "|r|≥0.5\nnegative"),
]

with PdfPages(out_pdf) as pdf:
    for f in forcings:
        cols = [
            f"{f}_r_TE_all",  f"{f}_n_TE_all",
            f"{f}_r_lag_all", f"{f}_n_lag_all",
            f"{f}_r_TE_dom",  f"{f}_n_TE_dom",
            f"{f}_r_lag_dom", f"{f}_n_lag_dom"
        ]
        col_labels = [
            "r_TE\n(all)", "n_TE\n(all)",
            "r_lag\n(all)", "n_lag\n(all)",
            "r_TE\n(dom)", "n_TE\n(dom)",
            "r_lag\n(dom)", "n_lag\n(dom)"
        ]

        cell_data   = []
        cell_colors = []
        for _, row in wide_df.iterrows():
            r_cells  = [row["attribute"]] + \
                       [str(row[c]) if row[c] != "" else "-" for c in cols]
            # only color r columns, not n columns
            c_colors = ["#D9E1F2"] + \
                       [cell_color(row[c]) if "r_" in c and "n_" not in c
                        else "white" for c in cols]
            cell_data.append(r_cells)
            cell_colors.append(c_colors)

        all_col_labels = ["Attributes"] + col_labels

        fig = plt.figure(figsize=(18, 26))

        # tight gridspec, no gap
        gs = fig.add_gridspec(2, 1,
                              height_ratios=[1, 22],
                              hspace=0.0,
                              top=0.97, bottom=0.01,
                              left=0.01, right=0.99)

        #  legend axis 
        leg_ax = fig.add_subplot(gs[0])
        leg_ax.axis("off")
        leg_ax.set_title(
            f"Comparison Table — {f}   (All forcings vs Dominant forcing only)",
            fontsize=12, fontweight="bold", pad=6)

        n_leg = len(legend_items)
        box_w = 1.0 / n_leg
        box_h = 0.5
        y_box = 0.3
        for i, (color, label) in enumerate(legend_items):
            x = i * box_w
            leg_ax.add_patch(plt.Rectangle(
                (x + 0.005, y_box), box_w - 0.01, box_h,
                facecolor=color, edgecolor="grey", linewidth=0.8,
                transform=leg_ax.transAxes, clip_on=False))
            leg_ax.text(
                x + box_w / 2, y_box - 0.05, label,
                ha="center", va="top", fontsize=7.5,
                transform=leg_ax.transAxes)

        #  table axis 
        tbl_ax = fig.add_subplot(gs[1])
        tbl_ax.axis("off")

        table = tbl_ax.table(
            cellText=cell_data,
            colLabels=all_col_labels,
            cellColours=cell_colors,
            cellLoc="center",
            loc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(7)
        table.scale(1, 1.25)

        # style header row
        for j in range(len(all_col_labels)):
            table[0, j].set_facecolor("#4472C4")
            table[0, j].set_text_props(color="white", fontweight="bold")

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("PDF saved:", out_pdf)

CSV saved: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\comparison_table_all_vs_dominant.csv
Shape: (50, 33)
PDF saved: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\pdf\comparison_table_all_vs_dominant.pdf


## Interpretation — Comparison Table (All Forcings vs Dominant Forcing Only)

### General observation
When we focus only on dominant basins, most r values get stronger compared
to the all-forcings analysis. This is expected — dominant basins are where
that forcing most clearly governs streamflow, so the attribute-TE relationship
is less noisy.

---

### PRCP (all basins = 670, dominant basins = 389)

**Peak TE:**
Water availability attributes remain the strongest predictors in both analyses.
runoff_ratio strengthens from +0.441 to +0.600,
low_prec_freq from -0.405 to -0.595,
high_prec_freq from -0.368 to -0.572.
Basins with higher runoff and less frequent dry or wet events
show stronger PRCP TE, and this pattern is clearer in dominant basins.

frac_snow changes sign from -0.112 (all) to +0.141 (dominant).
Across all basins, snowy basins tend to have lower PRCP TE because
energy forcings dominate there. But within PRCP-dominant basins only,
slightly more snow is still associated with marginally stronger PRCP TE.

**Peak lag:**
elev_mean drops from +0.326 (all) to +0.049 (dominant).
Across all basins, higher elevation is associated with longer PRCP lag,
partly because high-elevation snowy basins (where PRCP is not dominant)
have long lags. Once we restrict to PRCP-dominant basins only,
that contrast disappears and elevation no longer explains PRCP lag.
Most other lag predictors are also weak within PRCP-dominant basins (all below 0.16),
suggesting PRCP lag is not strongly governed by any single basin attribute
within this group.

---

### SRAD (all basins = 653, dominant basins = 156)

**Peak TE:**
slope_mean strengthens from +0.505 to +0.675 — the strongest predictor.
frac_snow strengthens from +0.382 to +0.550.
hfd_mean jumps from near zero (+0.017) to +0.416.
SRAD-dominant basins tend to be steeper, snowier, and have
later-season streamflow — consistent with mountain basins where
solar radiation drives snowmelt.

**Peak lag:**
p_seasonality strengthens from +0.327 to +0.475.
soil_porosity strengthens from +0.184 to +0.361.
gvf_diff strengthens from +0.305 to +0.332.
Basins with stronger precipitation seasonality, higher soil porosity,
and more vegetation seasonality show longer SRAD lag.
sand_frac becomes more negative (-0.181 to -0.364),
suggesting sandier soils are associated with shorter SRAD lag
in dominant basins — possibly because sandy soils transmit
water more quickly.

---

### Tair (all basins = 593, dominant basins = 95)

**Peak TE:**
The largest changes are seen here.
frac_snow jumps from +0.480 to +0.710,
elev_mean from +0.342 to +0.655,
hfd_mean from +0.148 to +0.583.
Tair-dominant basins are clearly high-elevation, snow-dominated,
with late-season streamflow — pointing to snowmelt as the
physical mechanism linking temperature to streamflow.

Vegetation seasonality attributes become strongly negative —
gvf_diff from -0.200 to -0.511, gvf_max from -0.029 to -0.470,
lai_diff from -0.114 to -0.395.
Tair-dominant basins tend to have low vegetation seasonality,
consistent with cold high-elevation environments.

p_mean changes sign from +0.217 (all) to -0.140 (dominant).
Within Tair-dominant basins, higher mean precipitation is not
associated with stronger Tair TE — snow fraction and elevation
are more important than overall wetness in this group.

**Peak lag:**
frac_snow strengthens from -0.407 to -0.577,
hfd_mean from -0.281 to -0.570.
Snow-dominated basins with early half-flow dates show
shorter Tair lag — meaning temperature drives streamflow
at shorter lags in those basins, consistent with a direct
snowmelt response to temperature during the melt season.

---

### VP (all basins = 577, dominant basins = 31)

**Note: VP has only 31 dominant basins — all results should be
interpreted with caution as r values are less stable with small samples.**

**Peak TE:**
frac_snow jumps from +0.480 to +0.758 — the strongest single r value
in the entire table. p_seasonality strengthens from -0.249 to -0.502.
The pattern is similar to Tair — VP-dominant basins tend to be
snow-dominated and strongly seasonal.

**Peak lag:**
clay_frac becomes strongly negative (-0.463),
hfd_mean strengthens (-0.308 to -0.455).
These patterns are suggestive but should not be over-interpreted
given the small sample size.

---

### Overall summary

| Forcing | Strongest r_TE change (all → dom) | Direction |
|---------|-----------------------------------|-----------|
| PRCP    | low_prec_freq: -0.405 → -0.595   | stronger  |
| SRAD    | slope_mean: +0.505 → +0.675      | stronger  |
| Tair    | frac_snow: +0.480 → +0.710       | stronger  |
| VP      | frac_snow: +0.480 → +0.758       | stronger  |

| Forcing | Strongest r_lag change (all → dom)  | Direction |
|---------|-------------------------------------|-----------|
| PRCP    | elev_mean: +0.326 → +0.049         | weaker    |
| SRAD    | p_seasonality: +0.327 → +0.475     | stronger  |
| Tair    | frac_snow: -0.407 → -0.577         | stronger  |
| VP      | hfd_mean: -0.308 → -0.455          | stronger  |


## Top 10 attribute rankings — Peak TE and Peak Lag, All vs Dominant

We rank attributes by absolute r value for each forcing.
This gives a quick summary of which attributes best explain
peak TE strength and peak lag across both analyses.

In [19]:
# load both correlation CSVs
all_csv = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\correlation_summary_TE_lag_vs_attributes.csv"

dom_csv = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\correlation_summary_dominant_forcing.csv"

all_df = pd.read_csv(all_csv)
dom_df = pd.read_csv(dom_csv)

forcings = ["PRCP", "SRAD", "Tair", "VP"]

# add absolute r columns
all_df["abs_r_peak_TE"]  = all_df["r_peak_TE"].abs()
all_df["abs_r_peak_lag"] = all_df["r_peak_lag"].abs()
dom_df["abs_r_peak_TE"]  = dom_df["r_peak_TE"].abs()
dom_df["abs_r_peak_lag"] = dom_df["r_peak_lag"].abs()

# Peak TE rankings 
print("=" * 65)
print("TOP 10 ATTRIBUTES BY |r| FOR PEAK TE — ALL FORCINGS")
print("=" * 65)
for f in forcings:
    sub = all_df[all_df["forcing"] == f].sort_values(
        "abs_r_peak_TE", ascending=False).head(10)
    n = int(sub["n_peak_TE"].max())
    print(f"\n--- {f} (n={n}) ---")
    for i, row in enumerate(sub.itertuples(), 1):
        print(f"  {i:2}. {row.attribute:<30} r = {row.r_peak_TE:+.3f}")

print("\n")
print("=" * 65)
print("TOP 10 ATTRIBUTES BY |r| FOR PEAK TE — DOMINANT FORCING")
print("=" * 65)
for f in forcings:
    sub = dom_df[dom_df["forcing"] == f].sort_values(
        "abs_r_peak_TE", ascending=False).head(10)
    n = int(sub["n_peak_TE"].max())
    print(f"\n--- {f} (n={n}) ---")
    for i, row in enumerate(sub.itertuples(), 1):
        print(f"  {i:2}. {row.attribute:<30} r = {row.r_peak_TE:+.3f}")

# Peak Lag rankings
print("\n")
print("=" * 65)
print("TOP 10 ATTRIBUTES BY |r| FOR PEAK LAG — ALL FORCINGS")
print("=" * 65)
for f in forcings:
    sub = all_df[all_df["forcing"] == f].sort_values(
        "abs_r_peak_lag", ascending=False).head(10)
    n = int(sub["n_peak_lag"].max())
    print(f"\n--- {f} (n={n}) ---")
    for i, row in enumerate(sub.itertuples(), 1):
        print(f"  {i:2}. {row.attribute:<30} r = {row.r_peak_lag:+.3f}")

print("\n")
print("=" * 65)
print("TOP 10 ATTRIBUTES BY |r| FOR PEAK LAG — DOMINANT FORCING")
print("=" * 65)
for f in forcings:
    sub = dom_df[dom_df["forcing"] == f].sort_values(
        "abs_r_peak_lag", ascending=False).head(10)
    n = int(sub["n_peak_lag"].max())
    print(f"\n--- {f} (n={n}) ---")
    for i, row in enumerate(sub.itertuples(), 1):
        print(f"  {i:2}. {row.attribute:<30} r = {row.r_peak_lag:+.3f}")

print("\nDone.")

TOP 10 ATTRIBUTES BY |r| FOR PEAK TE — ALL FORCINGS

--- PRCP (n=670) ---
   1. q95                            r = +0.624
   2. q_mean                         r = +0.612
   3. p_mean                         r = +0.588
   4. runoff_ratio                   r = +0.441
   5. low_prec_freq                  r = -0.405
   6. q5                             r = +0.399
   7. high_prec_freq                 r = -0.368
   8. p_seasonality                  r = -0.366
   9. hfd_mean                       r = -0.360
  10. aridity                        r = -0.337

--- SRAD (n=653) ---
   1. q95                            r = +0.603
   2. runoff_ratio                   r = +0.574
   3. q_mean                         r = +0.517
   4. slope_mean                     r = +0.505
   5. p_seasonality                  r = -0.480
   6. high_prec_freq                 r = -0.480
   7. low_prec_freq                  r = -0.404
   8. frac_snow                      r = +0.382
   9. gvf_diff                       r =

## CONUS map — Peak TE by forcing (significant only)

Same style as the Lag-1 TE and Weighted TE\* maps. Each basin is colored by its Peak TE percentile group (5 groups from Very Low to Very High). Basin counts per forcing now match the other two analyses (PRCP 670, SRAD 653, Tair 593, VP 577).

In [24]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io.shapereader import natural_earth, Reader

# project + output paths
project_folder = Path(r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes")
out_pdf  = project_folder / "outputs" / "pdf"
out_csv  = project_folder / "outputs" / "summary_csv"

# inputs
attr_path = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\camels_attributes_combined_671basins.csv"
te_path   = Path(r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\ALL CAMELS BASIN (FORCINGS TO STREAMFLOW)\outputs\summary_csv\TE_shuffle_all671_tau1_30_PRCPspecial_bins5.csv")

# load attributes (for lat/lon)
attr_df = pd.read_csv(attr_path, dtype={"gauge_id": str})
attr_df["gauge_id"] = attr_df["gauge_id"].str.zfill(8)
latlon = attr_df[["gauge_id", "gauge_lat", "gauge_lon"]].copy()

# load raw TE shuffle file and rebuild Peak TE using significant rows only
te_df = pd.read_csv(te_path, dtype={"gauge_id": str})
te_df["gauge_id"] = te_df["gauge_id"].str.zfill(8)

te_sig = te_df[te_df["sig"] == True].copy()
peak_idx = te_sig.groupby(["gauge_id", "Source"])["TE_obs"].idxmax()
peak_long = te_sig.loc[peak_idx, ["gauge_id", "Source", "Lag", "TE_obs"]].copy()
peak_long.columns = ["gauge_id", "Source", "peak_lag_sig", "peak_TE_sig"]

peak_TE_wide = peak_long.pivot(index="gauge_id", columns="Source", values="peak_TE_sig")
peak_TE_wide.columns = [f"peak_TE_sig_{c}" for c in peak_TE_wide.columns]

peak_sig_df = peak_TE_wide.reset_index()
peak_sig_df["gauge_id"] = peak_sig_df["gauge_id"].astype(str).str.zfill(8)

forcings = ["PRCP", "SRAD", "Tair", "VP"]
proj = ccrs.LambertConformal(central_longitude=-96, central_latitude=37.5)

# ----- helper functions -----
def draw_map_base(ax):
    ax.set_extent([-122, -68, 23, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="whitesmoke")
    ax.add_feature(cfeature.OCEAN, facecolor="#d4eaf7")
    ax.add_feature(cfeature.LAKES, facecolor="#d4eaf7")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.9, edgecolor="black")
    states = cfeature.NaturalEarthFeature(
        category="cultural",
        name="admin_1_states_provinces_lines",
        scale="50m", facecolor="none")
    ax.add_feature(states, edgecolor="grey", linewidth=0.5)

def add_state_names(ax):
    shpfile = natural_earth(resolution="50m", category="cultural",
                            name="admin_1_states_provinces")
    reader = Reader(shpfile)
    for record in reader.records():
        if record.attributes["admin"] != "United States of America":
            continue
        name = record.attributes["name"]
        geom = record.geometry
        cx, cy = geom.centroid.x, geom.centroid.y
        if cx < -122 or cx > -68 or cy < 23 or cy > 50:
            continue
        ax.text(cx, cy, name, fontsize=5, ha="center", va="center",
                color="dimgrey", transform=ccrs.PlateCarree(),
                fontweight="bold", zorder=4)

def add_title_box(ax, text):
    ax.text(0.02, 0.97, text, transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="top", ha="left",
            bbox=dict(facecolor="white", edgecolor="grey",
                      boxstyle="round,pad=0.3", alpha=0.85),
            zorder=10)

def add_vertical_legend(ax, colors, labels, title):
    handles = [mpatches.Patch(color=colors[g], label=labels[g])
               for g in range(len(colors))]
    ax.legend(handles=handles, loc="upper left",
              bbox_to_anchor=(1.01, 1.0), borderaxespad=0,
              fontsize=10, title=title, title_fontsize=11,
              framealpha=0.9, edgecolor="grey",
              handlelength=2.0, handleheight=2.0)

# ----- Peak TE CONUS map -----
te_bin_labels = ["Very Low", "Low", "Medium", "High", "Very High"]
te_bin_colors = ["#4a0080", "#66c266", "#ffd700", "#ff8c00", "#8b0000"]
pct_cuts      = [0, 20, 40, 60, 80, 100]

def get_te_bin(val, edges):
    for i in range(len(edges) - 1):
        if edges[i] <= val <= edges[i + 1]:
            return i
    return len(te_bin_labels) - 1

peakte_df = peak_sig_df.merge(latlon, on="gauge_id", how="left")

out_peakte_path = out_pdf / "CONUS_maps_peak_TE_significant.pdf"

with PdfPages(out_peakte_path) as pdf:
    fig, axes = plt.subplots(2, 2, figsize=(24, 15),
                             subplot_kw={"projection": proj})
    fig.suptitle("Peak TE by Forcing — 671 CAMELS Basins (significant only)",
                 fontsize=15, fontweight="bold", y=1.01)
    axes = axes.flatten()

    for ax, forcing in zip(axes, forcings):
        te_col = f"peak_TE_sig_{forcing}"
        sub = peakte_df[["gauge_lat", "gauge_lon", te_col]].dropna()

        vals  = sub[te_col].values
        edges = [np.percentile(vals, p) for p in pct_cuts]
        groups = sub[te_col].apply(lambda v: get_te_bin(v, edges))

        counts = [int((groups == g).sum()) for g in range(len(te_bin_labels))]

        draw_map_base(ax)
        add_state_names(ax)

        for g in range(len(te_bin_labels)):
            mask = groups == g
            s = sub[mask]
            if len(s) > 0:
                ax.scatter(s["gauge_lon"], s["gauge_lat"],
                           color=te_bin_colors[g],
                           s=20, alpha=0.9, zorder=5,
                           transform=ccrs.PlateCarree())

        add_title_box(ax, f"{forcing}  |  n = {len(sub)}")

        pct_ranges = ["0–20th", "20–40th", "40–60th", "60–80th", "80–100th"]
        te_labels_full = [
            f"{te_bin_labels[g]}\n({pct_ranges[g]} percentiles)\n"
            f"({edges[g]:.4f} – {edges[g+1]:.4f})\n"
            f"n = {counts[g]}"
            for g in range(len(te_bin_labels))
        ]
        add_vertical_legend(ax, te_bin_colors, te_labels_full, "Peak TE group")

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print("Done. Peak TE CONUS map saved to:")
print(out_peakte_path)

Done. Peak TE CONUS map saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\pdf\CONUS_maps_peak_TE_significant.pdf


## CONUS map — Peak Lag by forcing (significant only)

In [25]:
# project + output paths
project_folder = Path(r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes")
out_pdf  = project_folder / "outputs" / "pdf"

# inputs
attr_path = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\camels_attributes_combined_671basins.csv"
te_path   = Path(r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\ALL CAMELS BASIN (FORCINGS TO STREAMFLOW)\outputs\summary_csv\TE_shuffle_all671_tau1_30_PRCPspecial_bins5.csv")

# load attributes (for lat/lon)
attr_df = pd.read_csv(attr_path, dtype={"gauge_id": str})
attr_df["gauge_id"] = attr_df["gauge_id"].str.zfill(8)
latlon = attr_df[["gauge_id", "gauge_lat", "gauge_lon"]].copy()

# load raw TE shuffle file and rebuild Peak Lag using significant rows only
te_df = pd.read_csv(te_path, dtype={"gauge_id": str})
te_df["gauge_id"] = te_df["gauge_id"].str.zfill(8)

te_sig = te_df[te_df["sig"] == True].copy()
peak_idx = te_sig.groupby(["gauge_id", "Source"])["TE_obs"].idxmax()
peak_long = te_sig.loc[peak_idx, ["gauge_id", "Source", "Lag", "TE_obs"]].copy()
peak_long.columns = ["gauge_id", "Source", "peak_lag_sig", "peak_TE_sig"]

peak_lag_wide = peak_long.pivot(index="gauge_id", columns="Source", values="peak_lag_sig")
peak_lag_wide.columns = [f"peak_lag_sig_{c}" for c in peak_lag_wide.columns]

peak_lag_sig_df = peak_lag_wide.reset_index()
peak_lag_sig_df["gauge_id"] = peak_lag_sig_df["gauge_id"].astype(str).str.zfill(8)

forcings = ["PRCP", "SRAD", "Tair", "VP"]
proj = ccrs.LambertConformal(central_longitude=-96, central_latitude=37.5)

# ----- helper functions -----
def draw_map_base(ax):
    ax.set_extent([-122, -68, 23, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="whitesmoke")
    ax.add_feature(cfeature.OCEAN, facecolor="#d4eaf7")
    ax.add_feature(cfeature.LAKES, facecolor="#d4eaf7")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.9, edgecolor="black")
    states = cfeature.NaturalEarthFeature(
        category="cultural",
        name="admin_1_states_provinces_lines",
        scale="50m", facecolor="none")
    ax.add_feature(states, edgecolor="grey", linewidth=0.5)

def add_state_names(ax):
    shpfile = natural_earth(resolution="50m", category="cultural",
                            name="admin_1_states_provinces")
    reader = Reader(shpfile)
    for record in reader.records():
        if record.attributes["admin"] != "United States of America":
            continue
        name = record.attributes["name"]
        geom = record.geometry
        cx, cy = geom.centroid.x, geom.centroid.y
        if cx < -122 or cx > -68 or cy < 23 or cy > 50:
            continue
        ax.text(cx, cy, name, fontsize=5, ha="center", va="center",
                color="dimgrey", transform=ccrs.PlateCarree(),
                fontweight="bold", zorder=4)

def add_title_box(ax, text):
    ax.text(0.02, 0.97, text, transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="top", ha="left",
            bbox=dict(facecolor="white", edgecolor="grey",
                      boxstyle="round,pad=0.3", alpha=0.85),
            zorder=10)

def add_vertical_legend(ax, colors, labels, title):
    handles = [mpatches.Patch(color=colors[g], label=labels[g])
               for g in range(len(colors))]
    ax.legend(handles=handles, loc="upper left",
              bbox_to_anchor=(1.01, 1.0), borderaxespad=0,
              fontsize=10, title=title, title_fontsize=11,
              framealpha=0.9, edgecolor="grey",
              handlelength=2.0, handleheight=2.0)

# ----- Peak Lag CONUS map -----
lag_bin_edges  = [0, 2, 6, 11, 18, 30]
lag_bin_labels = ["1–2 (fast)", "3–6", "7–11", "12–18", "19–30 (slow)"]
lag_bin_colors = ["#8b0000", "#ff8c00", "#ffd700", "#66c266", "#4a0080"]

def get_lag_bin(val):
    for i in range(len(lag_bin_edges) - 1):
        if lag_bin_edges[i] < val <= lag_bin_edges[i + 1]:
            return i
    return len(lag_bin_labels) - 1

peaklag_df = peak_lag_sig_df.merge(latlon, on="gauge_id", how="left")

out_peaklag_path = out_pdf / "CONUS_maps_peak_lag_significant.pdf"

with PdfPages(out_peaklag_path) as pdf:
    fig, axes = plt.subplots(2, 2, figsize=(24, 15),
                             subplot_kw={"projection": proj})
    fig.suptitle("Peak Lag by Forcing — 671 CAMELS Basins (significant only)",
                 fontsize=15, fontweight="bold", y=1.01)
    axes = axes.flatten()

    for ax, forcing in zip(axes, forcings):
        lag_col = f"peak_lag_sig_{forcing}"
        sub = peaklag_df[["gauge_lat", "gauge_lon", lag_col]].dropna()
        groups = sub[lag_col].apply(get_lag_bin)

        counts = [int((groups == g).sum()) for g in range(len(lag_bin_labels))]

        draw_map_base(ax)
        add_state_names(ax)

        for g in range(len(lag_bin_labels)):
            mask = groups == g
            s = sub[mask]
            if len(s) > 0:
                ax.scatter(s["gauge_lon"], s["gauge_lat"],
                           color=lag_bin_colors[g],
                           s=20, alpha=0.9, zorder=5,
                           transform=ccrs.PlateCarree())

        add_title_box(ax, f"{forcing}  |  n = {len(sub)}")

        labels_with_n = [f"{lag_bin_labels[g]}  (n = {counts[g]})"
                         for g in range(len(lag_bin_labels))]
        add_vertical_legend(ax, lag_bin_colors, labels_with_n, "Peak lag (days)")

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print("Done. Peak Lag CONUS map saved to:")
print(out_peaklag_path)

Done. Peak Lag CONUS map saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Lag1_TE vs Attributes\outputs\pdf\CONUS_maps_peak_lag_significant.pdf


### Peak-lag map interpretation

This map shows the peak lag (the lag where TE is strongest, in days) for each basin, separately for each forcing. Only significant TE values are used. Basin counts per forcing: PRCP 670, SRAD 653, Tair 593, VP 577. The fast-vs-slow split across CONUS is clearly visible.

**PRCP — overwhelmingly fast.**
598 of 670 basins (89%) peak at lag 1–2 days. The map is almost solid dark red everywhere. Rainfall reaches streamflow within one to two days in nearly every basin across CONUS. Only 21 basins are slow (19–30 days), scattered with no clear region. PRCP is consistently the fast forcing.

**SRAD — mostly fast with a clear slow tail.**
432 fast (66%), 103 slow (16%). The slow SRAD basins cluster in the **Northeast** (purple points around New York, Pennsylvania, New Jersey, and the Mid-Atlantic). Solar radiation drives a quick response in most of CONUS, but in the cooler, wetter Northeast it acts more slowly.

**Tair — mostly slow.**
Only 119 fast (20%), and 234 slow (39%). This is the opposite of PRCP. The slow Tair basins dominate the **eastern half** — Northeast, Ohio Valley, Southeast — and parts of the central US. Temperature information takes weeks to peak in most basins, consistent with slow processes like snowmelt timing and seasonal ET rather than a direct daily response. The fast Tair basins (red) are mostly in the **Pacific Northwest and mountain West**, where snowmelt can respond quickly to a warm day.

**VP — between Tair and SRAD.**
195 fast (34%), 152 slow (26%). VP sits between the fast rainfall forcing and the slow temperature forcing. Slow VP basins again cluster in the **Northeast and Great Lakes**, while the West has more fast basins.

**The big pattern.**
For the energy forcings (SRAD, Tair, VP), the **West responds fast and the Northeast/Upper Midwest responds slow.** PRCP is fast almost everywhere, regardless of region. So the geographic fast-vs-slow divide is mostly a story about temperature-driven and humidity-driven forcings, not rainfall.

**Caution.**
The peak lag tells us when the TE signal is strongest, but it does not tell us *why*. A slow peak lag is consistent with snowmelt, storage, or seasonal ET, but the map alone does not prove which mechanism is at work. The basin-attribute correlations (next steps) help narrow this down by linking peak lag to attributes like frac_snow, slope, elevation, and aridity.